# LlamaIndex: 03 Retrieval & Query Engines

In this lab, we'll dive deep into retrievers and how to customize the answer synthesis.

In [1]:
import os
import nest_asyncio
from dotenv import load_dotenv
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Settings, VectorStoreIndex, Document

In [2]:
load_dotenv()
nest_asyncio.apply()

In [3]:
Settings.llm = GoogleGenAI(model="models/gemini-2.0-flash", api_key=os.getenv("GEMINI_API_KEY"))
Settings.embed_model = GoogleGenAIEmbedding(model_name="models/gemini-embedding-001", api_key=os.getenv("GEMINI_API_KEY"))

In [4]:
facts= [
    Document(text= "The secret password for the vault is: APPLEPIE."),
    Document(text= "The vault was built in 1985 by a team of Belgian architects."),
    Document(text= "The vault contains precisely 42 golden bars.")
]

In [5]:
index= VectorStoreIndex.from_documents(facts)

In [6]:
retriever= index.as_retriever(similarity_k_top= 2)

results= retriever.retrieve("What is the secret password?")

print(f"Total results found: {len(results)}")
for i, res in enumerate(results):
    print(f"\n--- Result {i} (Score: {res.score: .4f}) ---")
    print(f"Text: {res.node.get_content()}")

Total results found: 2

--- Result 0 (Score:  0.7891) ---
Text: The secret password for the vault is: APPLEPIE.

--- Result 1 (Score:  0.6218) ---
Text: The vault contains precisely 42 golden bars.


In [7]:
query_engine= index.as_query_engine()

response= query_engine.query("What is inside the vault and who built it?")

print(f"Answer: {response}")

print("\n--- The Evidence (Sources) ---")
for node in response.source_nodes:
    print(f"- {node.node.get_content()} (Score: {node.score: .4f})")

Answer: I do not have enough information to answer what is inside the vault. The vault was built by a team of Belgian architects in 1985.


--- The Evidence (Sources) ---
- The vault was built in 1985 by a team of Belgian architects. (Score:  0.7592)
- The secret password for the vault is: APPLEPIE. (Score:  0.7201)


In [8]:
smarter_engine= index.as_query_engine(similarity_top_k= 3)

response= smarter_engine.query("What is inside the vault and who built it?")
print(f"Smarter Answer: {response}")

Smarter Answer: The vault contains 42 golden bars and was built by a team of Belgian architects.

